# Project 1: Bangla and Banglish Text-to-SQL Robustness Benchmark

## Setup

In [ ]:

%pip install -q "chromadb>=1.0" "sentence-transformers>=3.0" "transformers>=4.44" \
                "pandas" "pyarrow" "tqdm" "openai>=1.0" "gdown>=5.0" "statsmodels"


In [ ]:
from pathlib import Path
import shutil
import zipfile

# -----------------------------------------------------------------------------
# Dataset auto-download (ZIP version)
# -----------------------------------------------------------------------------

# Google Drive FILE ID (not folder ID)
GDRIVE_FILE_ID = "1JVYkBFWpT99fqqb2eq9BRTwpPRauTlyL"

ZIP_NAME = "dataset.zip"
FORCE_DOWNLOAD = False

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATASET_DEST = PROJECT_ROOT / "dataset"
ZIP_PATH = PROJECT_ROOT / ZIP_NAME

_REQUIRED = [
    "spider_bangla_clean.csv",
    "spider_bangla_validation_clean.csv",
    "tables.json",
    "spider_data/database",
]


def dataset_present(root):
    """Check both normal and nested dataset layouts."""

    # Normal layout
    if root.exists() and all((root / p).exists() for p in _REQUIRED):
        return True

    # Nested layout (dataset/dataset/...)
    nested = root / "dataset"
    if nested.exists() and all((nested / p).exists() for p in _REQUIRED):
        return True

    return False


# -----------------------------------------------------------------------------
# Download
# -----------------------------------------------------------------------------

if dataset_present(DATASET_DEST) and not FORCE_DOWNLOAD:
    print(f"Dataset already exists at {DATASET_DEST}")

else:
    import gdown

    # Remove previous extraction if forcing download
    if FORCE_DOWNLOAD:
        if ZIP_PATH.exists():
            ZIP_PATH.unlink()
        if DATASET_DEST.exists():
            shutil.rmtree(DATASET_DEST)

    # Download ZIP
    if not ZIP_PATH.exists():
        print("Downloading dataset ZIP...")

        gdown.download(
            id=GDRIVE_FILE_ID,
            output=str(ZIP_PATH),
            quiet=False,
            fuzzy=True,
        )

    # Clean extraction directory
    if DATASET_DEST.exists():
        shutil.rmtree(DATASET_DEST)

    DATASET_DEST.mkdir(parents=True, exist_ok=True)

    print("Extracting ZIP...")

    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATASET_DEST)

    # -------------------------------------------------------------------------
    # Flatten dataset/dataset/* -> dataset/*
    # -------------------------------------------------------------------------

    nested = DATASET_DEST / "dataset"

    if nested.exists():
        print("Found nested dataset folder. Flattening...")

        for item in nested.iterdir():
            shutil.move(str(item), str(DATASET_DEST / item.name))

        nested.rmdir()

    # -------------------------------------------------------------------------
    # Validate
    # -------------------------------------------------------------------------

    if dataset_present(DATASET_DEST):

        print("\n Dataset ready!")

        print("\nTop level:")
        for p in sorted(DATASET_DEST.iterdir()):
            print(" -", p.name)

        db_dir = DATASET_DEST / "spider_data" / "database"

        if db_dir.exists():
            print(f"\nDatabases: {len(list(db_dir.iterdir()))}")

    else:

        print("\nExtraction finished, but expected files were not found.")

        print("\nCurrent directory structure:\n")

        for p in DATASET_DEST.rglob("*"):
            print(p.relative_to(DATASET_DEST))

        raise RuntimeError(
            "\nDataset extraction incomplete. "
            "Check the directory structure printed above."
        )

## 1. Imports & global configuration

All experiment constants live in one place. The two knobs that matter for evaluation:

- `SAMPLE_N` — how many validation questions to run (default 100; the full dev split is 1,034).
- `EVAL_SET` — `"validation"` (held-out) or `"train"`.

In [ ]:
from __future__ import annotations

import os
import json
import re
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ---- paths -------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
def _find(names):
    for cand_dir in (PROJECT_ROOT, PROJECT_ROOT / "dataset"):
        for n in names:
            p = cand_dir / n
            if p.exists():
                return p
    return PROJECT_ROOT / names[0]

DATA_CSV     = _find(["spider_bangla_clean.csv"])                 # train split
VALIDATION_CSV = _find(["spider_bangla_validation_clean.csv"])    # held-out validation
SPIDER_DIR   = (PROJECT_ROOT / "spider_data").resolve() if (PROJECT_ROOT / "spider_data").resolve().exists() else (PROJECT_ROOT / "dataset" / "spider_data")
DB_DIR       = SPIDER_DIR / "database"          # sqlite files, one dir per db_id
TABLES_JSON  = _find(["tables.json"])
ARTIFACTS    = PROJECT_ROOT / "artifacts"
CHROMA_DIR   = ARTIFACTS / "chroma"             # persistent vector store
RES_CSV      = ARTIFACTS / "spider_eval_results.csv"
ARTIFACTS.mkdir(exist_ok=True)

# ---- experiment constants ---------------------------------------------
EMBED_MODEL = "BAAI/bge-m3"     # 8192 ctx, 1024-dim dense, 100+ languages incl. Bangla
EMBED_DIM   = 1024
BATCH_SIZE  = 32
SEED        = 42

SAMPLE_N    = 10
EVAL_SET    = "validation"
SLEEP_SECONDS = 0.0            # inter-request sleep; 429 backoff is handled in the runner

LANG_COLS = {
    "en":         "question_en",
    "bn":         "question_bn",
    "banglish":   "question_banglish",
    "code_mixed": "question_code_mixed",
}
LANGS = list(LANG_COLS)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"device      : {DEVICE}")
print(f"project root: {PROJECT_ROOT}")
print(f"SAMPLE_N    : {SAMPLE_N}")

## 2. Load the dataset + schema registry

In [ ]:
df = pd.read_csv(DATA_CSV).reset_index(drop=True)
df["qid"] = df.index
print(f"train rows: {len(df):,}   dbs: {df['db_id'].nunique()}   cols: {len(df.columns)}")

with open(TABLES_JSON, encoding="utf-8") as f:
    _raw_tables = json.load(f)

SCHEMA = {}
for e in _raw_tables:
    tabs_orig, tabs_nl = e["table_names_original"], e["table_names"]
    cols_orig, cols_nl = e["column_names_original"], e["column_names"]
    ctypes, pk = e["column_types"], set(e["primary_keys"])

    columns = {}
    for gidx, ((tidx, cname), (_, cname_nl)) in enumerate(zip(cols_orig, cols_nl)):
        if tidx < 0:
            continue
        columns[gidx] = {"gidx": gidx, "table_idx": tidx, "table": tabs_orig[tidx],
                         "name": cname, "name_nl": cname_nl, "type": ctypes[gidx],
                         "is_pk": gidx in pk}

    tables = {}
    for tidx, tname in enumerate(tabs_orig):
        tables[tidx] = {"table_idx": tidx, "name": tname, "name_nl": tabs_nl[tidx],
                        "col_gidx": [g for g, c in columns.items() if c["table_idx"] == tidx]}

    fks = [(a, b) for a, b in e["foreign_keys"] if a in columns and b in columns]
    SCHEMA[e["db_id"]] = {"db_id": e["db_id"], "tables": tables, "columns": columns, "fks": fks}

print(f"schema registry: {len(SCHEMA)} dbs | "
      f"tables {sum(len(s['tables']) for s in SCHEMA.values()):,} | "
      f"columns {sum(len(s['columns']) for s in SCHEMA.values()):,} | "
      f"FK pairs {sum(len(s['fks']) for s in SCHEMA.values()):,}")

In [ ]:
vdf = pd.read_csv(VALIDATION_CSV).reset_index(drop=True)
vdf["qid"] = np.arange(len(vdf)) + 10_000_000

print(f"validation rows: {len(vdf):,}   dbs: {vdf['db_id'].nunique()}")
print(f"overlap with train dbs: {len(set(vdf['db_id']) & set(df['db_id'].unique()))} (expect 0)")

_miss_tabs  = sorted(set(vdf["db_id"]) - set(SCHEMA))
_miss_sqlite = sorted(d for d in set(vdf["db_id"]) if not (DB_DIR / d / f"{d}.sqlite").exists())
print(f"validation dbs missing from tables.json: {len(_miss_tabs)} {_miss_tabs[:5]}")
print(f"validation dbs missing sqlite file     : {len(_miss_sqlite)} {_miss_sqlite[:5]}")

for lang, col in LANG_COLS.items():
    if lang != "en":
        print(f"  {lang:<10} identical-to-EN rows: {int((vdf[col].astype(str) == vdf['question_en']).sum()):,}")

## 3. Bangla schema glosses (for the bilingual schema variant)

In [ ]:
def parse_schema_string(s):
    """-> [(table_token, [column_token, ...]), ...] from schema_en / schema_bn"""
    if not isinstance(s, str) or not s.strip():
        return []
    body = re.sub(r"^\s*(Tables|টেবিল)\s*:\s*", "", s.strip())
    out = []
    for chunk in body.split(";"):
        if ":" not in chunk:
            continue
        tab, cols = chunk.split(":", 1)
        out.append((tab.strip(), [c.strip() for c in cols.split(",") if c.strip()]))
    return out

GLOSS_BN = {}
_rows = pd.concat([
    df.drop_duplicates("db_id")[["db_id", "schema_en", "schema_bn"]],
    vdf.drop_duplicates("db_id")[["db_id", "schema_en", "schema_bn"]],
]).drop_duplicates("db_id")
for _, r in _rows.iterrows():
    db = r["db_id"]
    if db not in SCHEMA:
        continue
    en, bn = parse_schema_string(r["schema_en"]), parse_schema_string(r["schema_bn"])
    if not en or not bn or len(en) != len(bn) or any(len(a[1]) != len(b[1]) for a, b in zip(en, bn)):
        continue
    reg = SCHEMA[db]
    by_name = {t["name"].lower(): tidx for tidx, t in reg["tables"].items()}
    tmap, cmap, bad = {}, {}, False
    for (et, ec), (bt, bc) in zip(en, bn):
        tidx = by_name.get(et.lower())
        if tidx is None:
            bad = True
            break
        tmap[tidx] = bt
        rcols = {reg["columns"][g]["name"].lower(): g for g in reg["tables"][tidx]["col_gidx"]}
        for ecn, bcn in zip(ec, bc):
            g = rcols.get(ecn.lower())
            if g is not None:
                cmap[g] = bcn
    if not bad:
        GLOSS_BN[db] = {"tables": tmap, "columns": cmap}

print(f"dbs with usable Bangla glosses: {len(GLOSS_BN)} / {len(_rows)}")


def gloss_bn_table(db, tidx):
    return GLOSS_BN.get(db, {}).get("tables", {}).get(tidx)


def gloss_bn_column(db, gidx):
    return GLOSS_BN.get(db, {}).get("columns", {}).get(gidx)

## 4. Schema preprocessing → one natural-language chunk per table

In [ ]:
def _fk_lines(reg, gidx):
    cols = reg["columns"]
    return [f"{cols[a]['table']}.{cols[a]['name']} references {cols[b]['table']}.{cols[b]['name']}"
            for a, b in reg["fks"] if gidx in (a, b)]


def build_chunks(schema_lang="en"):
    """One document per table. schema_lang in {'en', 'bilingual'}."""
    rows = []
    for db, reg in SCHEMA.items():
        cols = reg["columns"]
        for tidx, t in reg["tables"].items():
            col_lines = []
            for g in t["col_gidx"]:
                c = cols[g]
                line = f"  {c['name']} ({c['type']})"
                if c["is_pk"]:
                    line += " [primary key]"
                if schema_lang == "bilingual":
                    bn = gloss_bn_column(db, g)
                    if bn:
                        line += f" / {bn}"
                col_lines.append(line)

            parts = [f"Table: {t['name']}", f"Description: {t['name_nl']}"]
            if schema_lang == "bilingual":
                bn_t = gloss_bn_table(db, tidx)
                if bn_t:
                    parts.append(f"Bangla: {bn_t}")
            parts.append("Columns:")
            parts.extend(col_lines)
            fk = [l for sub in (_fk_lines(reg, g) for g in t["col_gidx"]) for l in sub]
            if fk:
                parts.append("Foreign keys:")
                parts.extend(f"  {l}" for l in fk)
            rows.append({"db_id": db, "table": t["name"], "text": "\n".join(parts)})
    return pd.DataFrame(rows)


chunks_en = build_chunks("en")
chunks_bn = build_chunks("bilingual")
print(f"en chunks: {len(chunks_en):,}   bilingual chunks: {len(chunks_bn):,}")

## 5. Embedding model — `BAAI/bge-m3`

In [ ]:
from sentence_transformers import SentenceTransformer

_st_model = SentenceTransformer(EMBED_MODEL, device=DEVICE)
_st_model.max_seq_length = 512


def encode(texts, batch_size=32):
    """L2-normalized dense embeddings with robust input handling."""
    if isinstance(texts, str):
        texts = [texts]
    inputs = [str(t) for t in texts]
    embeddings = _st_model.encode(inputs, batch_size=batch_size,
                                  convert_to_numpy=True, show_progress_bar=False)
    if len(embeddings.shape) == 1:
        embeddings = embeddings.reshape(1, -1)
    norms = np.linalg.norm(embeddings, ord=2, axis=1, keepdims=True)
    return (embeddings / np.maximum(norms, 1e-12)).astype(np.float32)

print(f"Model loaded: {EMBED_MODEL} (dim={_st_model.get_sentence_embedding_dimension()})")

## 6. Index the schema into ChromaDB

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=str(CHROMA_DIR))


def index_chunks(name, chunks):
    col = client.get_or_create_collection(name, metadata={"hnsw:space": "cosine"})
    if col.count() >= len(chunks):
        print(f"{name}: already indexed ({col.count():,} chunks), skipping")
        return col
    docs  = chunks["text"].tolist()
    ids   = [f"{r.db_id}::{r.table}" for r in chunks.itertuples()]
    metas = chunks[["db_id", "table"]].to_dict("records")
    embs  = encode(docs).tolist()
    col.upsert(ids=ids, documents=docs, embeddings=embs, metadatas=metas)
    print(f"{name}: indexed {col.count():,} chunks")
    return col


col_en = index_chunks("schema_en", chunks_en)
col_bn = index_chunks("schema_bn", chunks_bn)
print(f"collection counts: {col_en.count():,} en / {col_bn.count():,} bn")

## 7. Retrieval — top-k schema for a question

In [ ]:
def retrieve(db, question, schema_lang="en", k=3):
    """-> DataFrame of top-k table chunks for `question` in `db`."""
    col = client.get_collection("schema_en" if schema_lang == "en" else "schema_bn")
    q_emb = encode([question])[0]
    res = col.query(query_embeddings=[q_emb.tolist()], n_results=k, where={"db_id": db})
    return pd.DataFrame({
        "table": [m["table"] for m in res["metadatas"][0]],
        "score": [round(1 - d, 4) for d in res["distances"][0]],
        "text":  res["documents"][0],
    })


row = vdf.iloc[0]   # demo row for the cells below
demo_top = retrieve(row["db_id"], row[LANG_COLS["bn"]], "en", k=3)
print(f"demo question ({row['db_id']}):\n{row[LANG_COLS['bn']]}\n")
print(demo_top[["table", "score"]].to_string(index=False))

## 8. Prompt construction

In [ ]:
def build_prompt(question, schema_text, lang="en", few_shot=None):
    fs = ""
    if few_shot:
        examples = "\n\n".join(
            f"Question: {q}\nSQL: {a}" for q, a in few_shot
        )
        fs = f"""
### Examples
{examples}
"""

    return f"""You are an expert SQLite text-to-SQL system.

Your task is to convert the user's question into exactly one valid SQLite SQL query.

You MUST follow these rules:

1. Use ONLY tables and columns explicitly present in the schema. Never invent names, relationships, or functions.
2. Use SQLite-compatible SQL only.
3. Return ONLY the SQL query - no explanations, comments, markdown, or code fences.

Value handling (string comparisons in SQLite are case-sensitive: 'Haiti' != 'haiti'):
4. For string literals, use the form STORED in the database, not the form displayed in the question.
   - If the schema shows sample values, copy the stored value verbatim (e.g. use 'haiti' if the sample shows 'haiti', even though the question says "Haiti").
   - If no sample value is shown and the case is uncertain, prefer lowercase.
5. Preserve numeric values from the question exactly as written.

Query construction:
6. Use JOIN conditions ONLY on relationships supported by the schema (foreign keys).
7. Filtering -> WHERE; aggregation (COUNT/SUM/AVG/MIN/MAX) -> GROUP BY when needed; highest/lowest -> ORDER BY ... LIMIT 1; a specific number of results -> LIMIT N; distinct values -> DISTINCT.
8. For percentage/ratio/average, construct the calculation carefully and avoid integer division (e.g. CAST(x AS REAL)).
9. When mixing AND and OR, use parentheses to make precedence explicit.
10. Do not add unnecessary tables, joins, filters, grouping, or ordering; prefer simple SQL.
11. Qualify ambiguous column names with the table name or alias.
12. Do not use information outside the schema and question.

### Schema
{schema_text}
{fs}
### User Question ({lang})
{question}

### Output
Return exactly one SQLite SQL query and nothing else."""


def full_schema_text(db):
    """The no-retrieval baseline prompt: every table of the db, in plain text."""
    reg = SCHEMA[db]
    lines = []
    for t in reg["tables"].values():
        cols = ", ".join(reg["columns"][g]["name"] for g in t["col_gidx"])
        lines.append(f"Table: {t['name']}\nColumns: {cols}")
    return "\n".join(lines)


print(build_prompt(row[LANG_COLS["bn"]],
                   "\n\n".join(demo_top["text"]), lang="bn")[:1500])

## 9. SQL generation — OpenAI-compatible API

In [ ]:
from openai import OpenAI
import os

base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
api_key = "" # api key removed 
LLM_MODEL = "gemma-4-31b-it"

llm = OpenAI(api_key=api_key, base_url=base_url)
print(f"LLM ready: {LLM_MODEL} via Google Generative Language")

import time
import re

def _extract_sql(content):
    """Robust SQL extraction for models that emit <thought> reasoning blocks
    (e.g. gemma-4-31b-it) and/or trailing prose.

    1. Drop <thought>...</thought> reasoning blocks.
    2. Prefer the last fenced ```sql ...``` block that actually contains SQL
       (reasoning may embed earlier fenced examples).
    3. Otherwise take the first SELECT statement, trimmed at trailing prose
       (backticks / "Alternatively:").
    """
    if not isinstance(content, str) or not content.strip():
        return ""
    s = re.sub(r"<thought>.*?</thought>", "", content, flags=re.DOTALL | re.IGNORECASE)
    fences = [f.strip() for f in re.findall(r"```(?:sql)?\s*(.*?)```", s, re.DOTALL | re.IGNORECASE) if f.strip()]
    sql_fences = [f for f in fences if re.search(r"SELECT\b", f, re.I)]
    if sql_fences:
        s = sql_fences[-1]
    else:
        m = re.search(r"SELECT\b", s, re.DOTALL | re.IGNORECASE)
        if not m:
            return ""
        cut = len(s)
        for pat in (r"`", r"alternatively"):
            mm = re.search(pat, s[m.start():], re.I)
            if mm:
                cut = min(cut, m.start() + mm.start())
        s = s[m.start():cut].strip()
    s = re.sub(r"^```(sql)?|```$", "", s, flags=re.M).strip()
    return s

def generate_sql(question, schema_text, lang='en', few_shot=None, model=LLM_MODEL):
    if llm is None:
        raise RuntimeError("LLM not configured \u2014 set GROQ_API_KEY (or OPENROUTER_API_KEY) and re-run cell 9.")
    resp = llm.chat.completions.create(
        model=model,
        temperature=0,
        messages=[{"role": "user", "content": build_prompt(question, schema_text, lang, few_shot)}],
    )
    return _extract_sql(resp.choices[0].message.content)

def generate_sql_with_retry(*args, **kwargs):
    max_retries = 6
    base_delay = 5
    for attempt in range(max_retries):
        try:
            return generate_sql(*args, **kwargs)
        except Exception as e:
            if "429" in str(e) or "rate limit" in str(e).lower():
                delay = base_delay * (2 ** attempt)
                print(f"\nRate limit hit. Retrying in {delay}s... (Attempt {attempt+1}/{max_retries})")
                time.sleep(delay)
            else:
                print(f"\nAPI Error: {e}")
                raise e
    raise RuntimeError("Max retries exceeded for API.")

## 10. Spider-faithful evaluation — EX and EM

**Methodology:**
- **Execution Accuracy (EX)**: Uses the **official Spider `evaluation.py`**, vendored verbatim into `spider_official/` (taoyds/spider master). Both queries are parsed into the Spider AST (`process_sql.py`), value- and FK-rebuilt exactly like `evaluate()`, then compared with `eval_exec_match`: per-SELECT-column value *lists* keyed by parsed val-units — row order and duplicates are significant, values compare with raw Python equality.
- **Exact Match (EM)**: We use a robust self-contained component match to avoid the dependency overhead of Spider's AST parser.
  > *Scientific Note: While our EM proxy is highly correlative, formal publications should swap this out for the official Spider `Evaluator().eval_exact_match` (already available in the vendored `spider_official/evaluation.py`) to guarantee exact comparability with SOTA papers.*


In [ ]:
def db_path(db):
    return DB_DIR / db / f"{db}.sqlite"


def _resolve_db(db):
    p = Path(db)
    if p.suffix == ".sqlite" and p.exists():
        return p
    return db_path(db)


def run_sql(db, sql):
    """-> dict(ok, rows, cols) or dict(ok=False, error). Read-only connection."""
    try:
        con = sqlite3.connect(f"file:{_resolve_db(db)}?mode=ro", uri=True)
        cur = con.execute(sql)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description] if cur.description else []
        con.close()
        return {"ok": True, "cols": cols, "rows": rows}
    except Exception as e:
        return {"ok": False, "error": str(e)}


# --- Official Spider EX: vendored taoyds/spider evaluation.py + process_sql.py ---
import sys as _sys
if str(PROJECT_ROOT) not in _sys.path:
    _sys.path.insert(0, str(PROJECT_ROOT))

from spider_official.evaluation import (
    eval_exec_match,
    build_foreign_key_map,
    build_valid_col_units,
    rebuild_sql_val,
    rebuild_sql_col,
)
from spider_official.process_sql import Schema, get_schema, get_sql

_EMPTY_SQL = {
    "except": None,
    "from": {"conds": [], "table_units": []},
    "groupBy": [],
    "having": [],
    "intersect": None,
    "limit": None,
    "orderBy": [],
    "select": [False, []],
    "union": None,
    "where": [],
}

_KMAPS = {e["db_id"]: build_foreign_key_map(e) for e in _raw_tables}

import re as _re

# Official Spider's parser requires `AS` before aliases; LLM preds often use
# bare aliases (FROM Professionals p). SQLite accepts both, the parser does
# not - so we insert `AS` (semantics-preserving) as a parse fallback.
_ALIAS_AHEAD = r"(?:AS|ON|USING|WHERE|JOIN|LEFT|RIGHT|INNER|OUTER|FULL|CROSS|NATURAL|GROUP|ORDER|HAVING|LIMIT|UNION|INTERSECT|EXCEPT|SET|VALUES|AND|OR)\b"
_BARE_ALIAS_RE = _re.compile(r"\b(FROM|JOIN)\s+([_a-zA-Z][\w]*)\s+([_a-zA-Z][\w]*)(?=\s+(?:" + _ALIAS_AHEAD + r")|\s*[,;)]|$)", _re.IGNORECASE)


def _add_as_aliases(sql):
    """Insert `AS` before bare table aliases so the official parser accepts the query."""
    return _BARE_ALIAS_RE.sub(r"\1 \2 AS \3", sql)


def exec_match_result(db, gold_sql, pred_sql):
    """Official Spider EX — taoyds/spider `evaluation.py` `eval_exec_match`.

    Both queries are parsed into the Spider AST (`process_sql.py`), value-normalized
    and FK-rebuilt exactly as `evaluate()` does, then executed. Two predictions match
    when their per-SELECT-column value lists are equal: row order AND duplicates are
    significant; keys are the parsed (unit_op, col_unit1, col_unit2) val_units; values
    compare with Python equality (1 == 1.0, 'Haiti' != 'haiti'). A prediction that
    fails to parse is scored as the official empty SQL (mismatch).

    Returns dict(exec_match, valid, gold_ok, error):
      - valid      -> True if pred_sql executes without error
      - gold_ok    -> True if gold_sql parses AND executes on this sqlite build
      - exec_match -> None when the gold itself fails (question not evaluable for EX)
    """
    try:
        schema = Schema(get_schema(str(_resolve_db(db))))
    except Exception as e:
        return {"exec_match": None, "valid": False, "gold_ok": False, "error": f"schema: {e}"}

    try:
        g_sql = get_sql(schema, gold_sql)
    except Exception as e:
        return {"exec_match": None, "valid": False, "gold_ok": False,
                "error": f"gold parse failed: {str(e)[:120]}"}

    try:
        p_sql = get_sql(schema, pred_sql)
    except Exception:
        try:
            p_sql = get_sql(schema, _add_as_aliases(pred_sql))
        except Exception:
            p_sql = dict(_EMPTY_SQL)

    kmap = _KMAPS.get(db, {})
    g_sql = rebuild_sql_val(g_sql)
    g_sql = rebuild_sql_col(build_valid_col_units(g_sql["from"]["table_units"], schema), g_sql, kmap)
    p_sql = rebuild_sql_val(p_sql)
    p_sql = rebuild_sql_col(build_valid_col_units(p_sql["from"]["table_units"], schema), p_sql, kmap)

    g_run = run_sql(db, gold_sql)
    if not g_run["ok"]:
        return {"exec_match": None, "valid": False, "gold_ok": False,
                "error": f"gold failed: {g_run['error'][:120]}"}

    match = eval_exec_match(str(_resolve_db(db)), pred_sql, gold_sql, p_sql, g_sql)
    valid, error = True, ""
    if not match:
        p_run = run_sql(db, pred_sql)
        valid, error = p_run["ok"], p_run.get("error", "")[:200]
    return {"exec_match": bool(match), "valid": valid, "gold_ok": True, "error": error}



# Sanity check on the demo row: gold must execute.
_demo_run = run_sql(row["db_id"], row["query"])
print("gold SQL ok:", _demo_run["ok"], "| rows:", len(_demo_run["rows"]) if _demo_run["ok"] else _demo_run["error"])


In [ ]:
def _em_tokens(sql):
    """Tokenize SQL: words, numbers, string literals, operators, parens, commas, dots."""
    pat = re.compile(r"[A-Za-z_][A-Za-z_0-9]*|\d+\.\d+|\d+|'[^']*'|>=|<=|!=|<>|=|<|>|\(|\)|,|\.", re.I)
    return [t.lower() for t in pat.findall(str(sql))]

_EM_KW = {
    "select", "from", "where", "group", "by", "order", "having", "limit", "on", "as", "and", "or",
    "not", "in", "between", "like", "distinct", "count", "sum", "avg", "min", "max", "asc", "desc",
    "union", "except", "intersect", "left", "right", "inner", "outer", "full", "cross", "join",
    "is", "null", "all", "exists", "case", "when", "then", "else", "end", "offset", "values",
    "top", "over", "partition",
}


def _split_top(tokens, seps):
    """Split a token list on top-level separator tokens (paren depth 0)."""
    out, depth, cur = [], 0, []
    for t in tokens:
        if t == "(":
            depth += 1
        elif t == ")":
            depth -= 1
        if depth == 0 and t in seps:
            if cur:
                out.append(cur)
                cur = []
        else:
            cur.append(t)
    if cur:
        out.append(cur)
    return out


def _is_num(t):
    try:
        float(t)
        return True
    except ValueError:
        return False


def _norm_num(t):
    f = float(t)
    return str(int(f)) if f.is_integer() else repr(f)


def _strip_quals(tokens):
    """Drop '<word> .' qualifiers so x.id == id (symmetric for aliased and bare SQL)."""
    out, i = [], 0
    while i < len(tokens):
        t = tokens[i]
        if i + 1 < len(tokens) and tokens[i + 1] == "." and t not in _EM_KW and not _is_num(t):
            i += 2
            continue
        out.append(t)
        i += 1
    return out


def _strip_alias(tokens):
    """Drop 'AS x' or a trailing bare alias word (applied symmetrically to both sides).
    A single remaining token is the column itself, never stripped."""
    if len(tokens) >= 3 and tokens[-2] == "as":
        tokens = tokens[:-2]
    elif len(tokens) >= 2 and tokens[-1] not in _EM_KW and tokens[-1] not in (")", "(", ",", ".") \
            and not _is_num(tokens[-1]):
        tokens = tokens[:-1]
    return tokens


def _norm_cond(tokens):
    """Canonicalize one condition: alias qualifiers stripped, numbers normalized,
    string literals -> <str>."""
    tokens = _strip_quals(tokens)
    return " ".join(_norm_num(t) if _is_num(t) else ("<str>" if t.startswith("'") else t)
                    for t in tokens)


def _norm_col(tokens):
    """Canonicalize a SELECT/GROUP BY/ORDER BY column: optional agg(expr), alias stripped."""
    tokens = _strip_quals(_strip_alias([t for t in tokens if t != ","]))
    if len(tokens) >= 3 and tokens[0] in ("count", "sum", "avg", "min", "max") and tokens[1] == "(":
        body = " ".join(tokens[2:-1]) if tokens[-1] == ")" else " ".join(tokens[2:])
        return f"{tokens[0]}({body})"
    return " ".join(tokens)


def _find_clause(tokens, start, name):
    """Top-level index of clause `name`: 'where' | 'group by' | 'order by' | 'having' | 'limit'."""
    depth, i = 0, start
    while i < len(tokens):
        t = tokens[i]
        if t == "(":
            depth += 1
        elif t == ")":
            depth -= 1
        if depth == 0:
            if name in ("group by", "order by") and t == name.split()[0] \
                    and i + 1 < len(tokens) and tokens[i + 1] == "by":
                return i
            if t == name:
                return i
        i += 1
    return None


def _clause_region(tokens, name, kw_len, clause_idx):
    """Token slice of clause `name` from `clause_idx` to the next clause start."""
    nexts = [_find_clause(tokens, 0, n) for n in ("group by", "order by", "having", "limit")]
    nexts = [n for n in nexts if n is not None and n > clause_idx]
    end = min(nexts) if nexts else len(tokens)
    return tokens[clause_idx + kw_len:end]


def extract_components(sql):
    """-> dict of Spider-style components, or None when unparsable.

    Components: distinct, set_ops, selects, from_tables, join_conds, wheres,
    groupbys, orderbys, havings, limit. All tokens lower-cased and aliases
    stripped symmetrically, so pred == gold means component match.
    """
    if not isinstance(sql, str) or not sql.strip():
        return None
    s = re.sub(r"--[^\n]*", "", sql)
    s = re.sub(r"/\*.*?\*/", "", s, flags=re.S)
    toks = [t for t in _em_tokens(s) if t != ";"]
    if not toks or toks[0] not in ("select", "("):
        return None

    comp = {"distinct": False, "selects": [], "from_tables": [], "join_conds": [],
            "wheres": [], "groupbys": [], "orderbys": [], "havings": [], "limit": None}

    set_ops = [t for t in toks if t in ("union", "except", "intersect")]
    if set_ops:
        comp["set_ops"] = set_ops
    segs = _split_top(toks, ["union", "except", "intersect"])

    for seg in segs:
        from_i = _find_clause(seg, 0, "from")

        # ---- SELECT columns ------------------------------------------
        if seg[:1] == ["select"]:
            sel_end = from_i if from_i is not None else len(seg)
            sel_toks = seg[1:sel_end]
            if sel_toks and sel_toks[0] == "distinct":
                comp["distinct"] = True
                sel_toks = sel_toks[1:]
            for col in _split_top(sel_toks, [","]):
                c = _norm_col(col)
                if c:
                    comp["selects"].append(c)

        if from_i is None:
            continue

        # ---- FROM tables + JOIN conditions ---------------------------
        rest = seg[from_i + 1:]
        frm_end = len(rest)
        for n, kw in (("group by", 2), ("order by", 2), ("having", 1), ("limit", 1)):
            i = _find_clause(rest, 0, n)
            if i is not None and i < frm_end:
                frm_end = i
        frm = rest[:frm_end]

        join_toks = [i for i, t in enumerate(frm) if t == "join"]
        table_pos = {0} | {i + 1 for i in join_toks} | {
            i for i in range(1, len(frm)) if frm[i] == "," and frm[i - 1] not in ("(", "on")}
        for i in sorted(table_pos):
            if i >= len(frm):
                continue
            t = frm[i]
            if t in _EM_KW or t in ("(", ")", ",", ".") or (i > 0 and frm[i - 1] == "."):
                continue
            comp["from_tables"].append(t)

        for j in join_toks:
            on_i = next((k for k in range(j + 1, len(frm)) if frm[k] == "on"), None)
            if on_i is None:
                continue
            end = next((k for k in range(on_i + 1, len(frm)) if frm[k] == "join"), len(frm))
            for c in _split_top(frm[on_i + 1:end], ["and", "or"]):
                comp["join_conds"].append(_norm_cond(c))

        # ---- WHERE / GROUP BY / ORDER BY / HAVING / LIMIT -------------
        for n, kw_len, dst in (("where", 1, "wheres"), ("having", 1, "havings")):
            i = _find_clause(rest, 0, n)
            if i is None:
                continue
            for c in _split_top(_clause_region(rest, n, kw_len, i), ["and", "or"]):
                comp[dst].append(_norm_cond(c))
        for n, kw_len, dst, split in (("group by", 2, "groupbys", [","]),
                                      ("order by", 2, "orderbys", [","])):
            i = _find_clause(rest, 0, n)
            if i is None:
                continue
            for c in _split_top(_clause_region(rest, n, kw_len, i), split):
                comp[dst].append(_norm_col(c))
        i = _find_clause(rest, 0, "limit")
        if i is not None:
            lim = _clause_region(rest, "limit", 1, i)
            if lim and _is_num(lim[0]):
                comp["limit"] = _norm_num(lim[0])

    return comp


def exact_match(gold_sql, pred_sql):
    """Spider-style component exact match. Unparsable prediction -> mismatch."""
    g, p = extract_components(gold_sql), extract_components(pred_sql)
    return bool(g and p and g == p)

## 10b. Sanity checks — official EX/EM semantics

Verify the vendored official metric (`eval_exec_match`) behaves as the official Spider protocol does:

1. **Duplicates are significant**: `SELECT name FROM t` vs `SELECT DISTINCT name FROM t` is a MISMATCH — official compares per-column value *lists*, not sets.
2. **Numeric equality**: `1` and `1.0` are equal (raw Python `==`).
3. **Row order is significant**: same rows in a different order (e.g. via `ORDER BY DESC`) is a MISMATCH.
4. **Value case is significant**: `'a'` vs `'A'` is a MISMATCH (no lowercasing).
5. **Unparseable predictions** are scored as the official empty SQL → mismatch, `valid=False`.
6. **EM** is symmetric and value-type-insensitive (string literal `'x'` == `'x'`), but breaks on real SQL changes.

In [ ]:
import tempfile, os

_tmp = Path(tempfile.gettempdir()) / "spider_ex_demo.sqlite"
if _tmp.exists():
    _tmp.unlink()
_con = sqlite3.connect(_tmp)
try:
    _con.execute("CREATE TABLE t(id INTEGER, name TEXT)")
    _con.executemany("INSERT INTO t VALUES (?,?)", [(1, 'a'), (1, 'a'), (2, 'b')])
    _con.commit()
finally:
    _con.close()


def _ex(gold_sql, pred_sql, label, expect):
    r = exec_match_result(_tmp, gold_sql, pred_sql)
    tag = "PASS" if r["exec_match"] == expect else "FAIL"
    print(f"{label:34s} exec_match={r['exec_match']} (expect {expect})  {tag}  valid={r['valid']}")

# 1) duplicates ARE significant: gold DISTINCT vs pred not -> official MISMATCH.
_ex("SELECT DISTINCT name FROM t", "SELECT name FROM t", "1) dup/DISTINCT", expect=False)
# 2) numeric literal normalization: WHERE id > 1.0 vs WHERE id > 1 -> official match (rebuild_sql_val).
_ex("SELECT id FROM t WHERE id > 1.0", "SELECT id FROM t WHERE id > 1", "2) numeric literal", expect=True)
# 3) row order IS significant: same rows, reversed order -> official MISMATCH.
_ex("SELECT name FROM t WHERE id <= 2", "SELECT name FROM t WHERE id <= 2 ORDER BY name DESC", "3) row order", expect=False)
# 4) value case IS significant: 'a' vs 'A' -> official MISMATCH (no lowercasing).
_ex("SELECT name FROM t WHERE name = 'a'", "SELECT name FROM t WHERE name = 'A'", "4) value case", expect=False)
# 5) unparseable prediction -> official empty SQL -> mismatch, valid=False.
_ex("SELECT DISTINCT name FROM t", "SELECT name FROOOM t WHERE", "5) parse failure", expect=False)

# 3) EM sanity
q1 = "SELECT DISTINCT name FROM t WHERE id > 1 ORDER BY name"
q2 = "SELECT DISTINCT name FROM t where id > 1 order by name"     # case/spacing only
q3 = "SELECT DISTINCT id FROM t WHERE id > 1 ORDER BY name"       # different column
q4 = "SELECT DISTINCT name FROM t AS x WHERE x.id > 1 ORDER BY name"  # alias-equivalent
q5 = "SELECT DISTINCT name FROM t WHERE id >= 1 ORDER BY name"    # different operator
q6 = "SELECT COUNT(*) FROM t"                                      # different shape
print(f"\nEM self-equal          : {exact_match(q1, q2)} (expect True)")
print(f"EM different column    : {exact_match(q1, q3)} (expect False)")
print(f"EM alias-equivalent    : {exact_match(q1, q4)} (expect True)")
print(f"EM different operator  : {exact_match(q1, q5)} (expect False)")
print(f"EM different shape     : {exact_match(q1, q6)} (expect False)")

# EM on the real gold query + a wrong but plausible prediction
print(f"\nEM gold==gold      : {exact_match(row['query'], row['query'])} (expect True)")

## 11. Experiment — full metric matrix, Spider

In [ ]:
import time

SQL_KEYWORDS = {
    "select", "from", "where", "group", "by", "order", "having", "limit", "join", "on", "as",
    "and", "or", "not", "in", "between", "like", "distinct", "count", "sum", "avg", "min", "max",
    "asc", "desc", "union", "intersect", "except", "left", "right", "inner", "outer", "full",
    "is", "null", "all", "exists", "case", "when", "then", "else", "end", "offset", "values",
    "t1", "t2", "t3", "t4", "t5",
}

def schema_links(db, sql):
    """-> dict(tables=set, columns=set of 'table.column') touched by `sql` (heuristic)."""
    reg = SCHEMA.get(db)
    if reg is None or not isinstance(sql, str):
        return {"tables": set(), "columns": set()}
    toks = re.findall(r"[A-Za-z_][A-Za-z_0-9]*", sql.lower())
    tbl_by_name = {t["name"].lower(): t["name"] for t in reg["tables"].values()}
    tables = {tbl_by_name[toks[i + 1]] for i, tok in enumerate(toks[:-1])
              if tok in ("from", "join") and toks[i + 1] in tbl_by_name}
    if not tables:
        tables = {tbl_by_name[t] for t in set(toks) & set(tbl_by_name)}
    cols_by_name = {}
    for c in reg["columns"].values():
        cols_by_name.setdefault(c["name"].lower(), []).append(c)
    columns = set()
    for tok in set(toks) - SQL_KEYWORDS:
        for c in cols_by_name.get(tok, []):
            if not tables or c["table"] in tables:
                columns.add(f"{c['table']}.{c['name']}")
    return {"tables": tables, "columns": columns}


def provided_columns(db, table_names):
    """All 'table.column' names of the given tables — what the prompt actually contained."""
    reg = SCHEMA.get(db)
    if reg is None:
        return set()
    tidx_by_name = {t["name"]: i for i, t in reg["tables"].items()}
    out = set()
    for tn in table_names:
        tidx = tidx_by_name.get(tn)
        if tidx is None:
            continue
        for g in reg["tables"][tidx]["col_gidx"]:
            c = reg["columns"][g]
            out.add(f"{c['table']}.{c['name']}")
    return out


FEW_SHOT_N = 2
FEW_SHOT_SRC_DB = "soccer_2"
MODELS = [LLM_MODEL]

CONFIGS = [
    (None, 0, False, False, False),        # baseline: full schema, zero-shot
    ("en", 3, False, False, False),        # retrieval, English schema, k=3
    ("en", 5, False, False, False),        # retrieval, English schema, k=5
    ("bilingual", 3, False, False, False), # retrieval, bilingual schema, k=3
    ("bilingual", 5, False, False, False), # retrieval, bilingual schema, k=5
    ("en", 3, True, False, False),         # retrieval + few-shot
    ("en", 3, False, True, False),         # retrieval + sample values
    ("en", 3, False, False, True),         # retrieval + value grounding (stored spellings)
    ("bilingual", 3, False, False, True),  # bilingual retrieval + value grounding
]

_few_pool = df[df.db_id == FEW_SHOT_SRC_DB]
FEW_SHOT = [(r.question_en, r.query) for r in _few_pool.head(FEW_SHOT_N).itertuples()] if len(_few_pool) else []


def cfg_label(cfg):
    sl, k, fs, vals, vg = cfg
    base = "baseline" if sl is None else f"retrieval_{sl}_k{k}"
    return base + ("_fs" if fs else "") + ("_vals" if vals else "") + ("_vground" if vg else "")


def sql_complexity(sql):
    s = str(sql).lower()
    nested = s.count("select") - 1
    if nested > 0 or any(w in s for w in ("intersect", "except", "union")):
        return "extra" if nested > 1 else "hard"
    if "join" in s or "group by" in s or "having" in s:
        return "medium"
    return "easy"


def sample_values_text(db, tables):
    reg = SCHEMA.get(db)
    if reg is None:
        return ""
    lines = []
    try:
        con = sqlite3.connect(f"file:{db_path(db)}?mode=ro", uri=True)
        by_name = {t["name"].lower(): t for t in reg["tables"].values()}
        for tname in tables:
            t = by_name.get(str(tname).lower())
            if not t:
                continue
            for g in t["col_gidx"]:
                c = reg["columns"][g]
                try:
                    rows = con.execute(f'SELECT DISTINCT "{c["name"]}" FROM "{c["table"]}" LIMIT 3').fetchall()
                    vals = [str(r[0]) for r in rows if r[0] is not None]
                    if vals:
                        lines.append(f"  {c['table']}.{c['name']}: {', '.join(vals)}")
                except Exception:
                    continue
        con.close()
    except Exception:
        pass
    return ("Sample values:\n" + "\n".join(lines)) if lines else ""




# --- Value grounding: report STORED spellings of values that appear in the question ---
_VG_CACHE = {}

def _col_values(db, table, column):
    key = f"{db}|{table}|{column}"
    if key in _VG_CACHE:
        return _VG_CACHE[key]
    out = []
    try:
        con = sqlite3.connect(f"file:{db_path(db)}?mode=ro", uri=True)
        rows = con.execute(f'SELECT DISTINCT "{column}" FROM "{table}" WHERE "{column}" IS NOT NULL LIMIT 500').fetchall()
        con.close()
        vals = [str(r[0]) for r in rows]
        if vals and not all(_is_number(v) for v in vals):
            out = vals
    except Exception:
        pass
    _VG_CACHE[key] = out
    return out


def _is_number(v):
    try:
        float(v)
        return True
    except ValueError:
        return False


_VG_STOPWORDS = {"the","a","an","is","are","was","were","of","in","on","at","by","for","with",
    "to","from","and","or","as","it","its","their","they","this","that","have","has","had","be",
    "been","which","who","whom","whose","what","how","many","much","do","does","did","not","no",
    "yes","all","each","every","both","most","more","less","than","over","under","between",
    "total","number","name","list","show","find","give","there","then","than","about","into",
    "through","during","before","after","above","below","up","down","out","off","only","also",
    "any","some","few","other","another","per","each","select","from","where"}


def _question_ngrams(question, max_words=3):
    words = [w.strip("'\"`") for w in re.findall(r"[A-Za-z0-9'\-]+", question)]
    words = [w for w in words if len(w) >= 3 and not w.isdigit()]
    out = []
    for n in range(1, max_words + 1):
        for i in range(len(words) - n + 1):
            g = " ".join(words[i:i + n])
            if len(g) <= 60:
                out.append(g)
    return out


def value_grounding(db, question, tables):
    """Find question tokens that match DB string values case-insensitively but NOT
    exactly. Returns a prompt note giving the STORED spelling for each."""
    reg = SCHEMA.get(db)
    if reg is None:
        return ""
    by_name = {t["name"].lower(): t for t in reg["tables"].values()}
    col_sets = []  # (column_name, table_name, {lower: stored})
    for tname in tables:
        t = by_name.get(str(tname).lower())
        if not t:
            continue
        for g in t["col_gidx"]:
            c = reg["columns"][g]
            vals = _col_values(db, c["table"], c["name"])
            if vals:
                col_sets.append((c["name"], c["table"], {v.lower(): v for v in vals}))
    if not col_sets:
        return ""
    notes, used = [], set()
    for gram in _question_ngrams(question):
        key = gram.lower()
        if key in used or key in _VG_STOPWORDS:
            continue
        for cname, tname, lookup in col_sets:
            if key in lookup:
                stored = lookup[key]
                if stored != gram:
                    notes.append(f"- '{gram}' is stored as '{stored}' in {tname}.{cname}")
                elif not gram.islower():
                    notes.append(f"- '{gram}' is stored exactly as written in {tname}.{cname}")
                used.add(key)
                break
        if len(notes) >= 10:
            break
    if not notes:
        return ""
    return ("Stored-value notes (from the database - use these EXACT spellings in string literals):\n"
            + "\n".join(notes))


# ---- stratified sample + resume from cache -----------------------------
def stratified_sample(df, n, seed):
    """Round-robin over databases: every db contributes one question before
    any db contributes a second. Deterministic via `seed`."""
    rng = np.random.default_rng(seed)
    pools = [g.index.tolist() for _, g in df.groupby("db_id")]
    rng.shuffle(pools)
    picked, i = [], 0
    while len(picked) < n and any(p for p in pools):
        p = pools[i % len(pools)]
        i += 1
        if p:
            picked.append(p.pop())
    return df.loc[picked]

# --- PHASE 1: English Configurations Elimination ---
EVAL_SAMPLE_N = 50  # 150 = full stratified run (~8-9h); 10 = quick config check
eval_df = vdf if EVAL_SET == "validation" else df
sample = stratified_sample(eval_df, min(EVAL_SAMPLE_N, len(eval_df)), SEED)
print(f"Phase 1 sample: {len(sample)} questions | {sample['db_id'].nunique()} dbs")

phase1_results = []
n_new = 0

print("\n--- Starting Phase 1: Evaluating all configs on English ---")
for r in tqdm(sample.itertuples(), total=len(sample), desc="Phase 1 (EN)"):
    gold_links = schema_links(r.db_id, r.query)
    lang = "en"
    question = r._asdict()[LANG_COLS[lang]]
    
    for cfg in CONFIGS:
        for model in MODELS:
            label = cfg_label(cfg)
            sl, k, fs, vals, vg = cfg
            if sl is None:
                schema_text = full_schema_text(r.db_id)
                pred_tables = {t["name"] for t in SCHEMA[r.db_id]["tables"].values()}
            else:
                top = retrieve(r.db_id, question, sl, k)
                schema_text, pred_tables = "\n\n".join(top["text"]), set(top["table"])
            if vals:
                schema_text += "\n\n" + sample_values_text(r.db_id, pred_tables)
            if vg:
                vg_note = value_grounding(r.db_id, question, pred_tables)
                if vg_note:
                    schema_text += "\n\n" + vg_note

            try:
                pred = generate_sql_with_retry(question, schema_text, lang=lang, few_shot=FEW_SHOT if fs else None, model=model)
                ev = exec_match_result(r.db_id, r.query, pred)
                pred_links = schema_links(r.db_id, pred)
                gt, gc, pc = gold_links["tables"], gold_links["columns"], pred_links["columns"]
                prov_cols = provided_columns(r.db_id, pred_tables)
                
                phase1_results.append({
                    "qid": r.qid, "db_id": r.db_id, "lang": lang, "config": label, "model": model,
                    "pred_sql": pred, "gold_sql": r.query, "question": question,
                    "complexity": sql_complexity(r.query),
                    "valid": ev["valid"], "gold_ok": ev["gold_ok"], "error": ev["error"],
                    "exec_match": ev["exec_match"],
                    "em_match": exact_match(r.query, pred),
                    "table_recall": len(gt) and len(gt & pred_tables) / len(gt) or 0,
                    "table_exact": float(bool(gt) and gt == pred_tables),
                    "col_recall": len(gc) and len(gc & prov_cols) / len(gc) or 0,
                    "col_recall_pred": len(gc) and len(gc & pc) / len(gc) or 0,
                })
            except Exception as e:
                print(f"Skipping due to error: {e}")
                continue
            if not ev["exec_match"]:
                print(f"\n[MISMATCH] QID: {r.qid}\nGold: {r.query}\nPred: {pred}\nError: {ev['error']}")
            
            if len(phase1_results) % 5 == 0:
                pd.DataFrame(phase1_results).to_csv(ARTIFACTS / "spider_eval_results_strategy_b_phase1_debug.csv", index=False)


# Determine Top 2 Configs
p1_df = pd.DataFrame(phase1_results)
p1_ok = p1_df[p1_df.gold_ok].copy()
config_scores = p1_ok.groupby("config")["exec_match"].mean().sort_values(ascending=False)
print("\nPhase 1 Results (English Execution Accuracy):")
print(config_scores)

top_2_labels = config_scores.head(2).index.tolist()
if not top_2_labels:
    print("Warning: No successful configs found in Phase 1. Proceeding with baseline only.")
    top_2_labels = ["baseline"]
# k=5 must always be scored on the Bangla variants (banglish recall is where k=5 can help),
# so it is guaranteed a Phase 2 slot even when EN scores tie.
if "retrieval_bilingual_k5" not in top_2_labels:
    top_2_labels.append("retrieval_bilingual_k5")

top_2_configs = [c for c in CONFIGS if cfg_label(c) in top_2_labels]
print(f"\nSelected Top configs for cross-lingual Phase 2: {top_2_labels}")

# --- PHASE 2: Cross-lingual on Top Configs ---
print("\n--- Starting Phase 2: Evaluating Top configs on Bangla variants ---")
phase2_results = []
CROSS_LANGS = [l for l in LANGS if l != "en"]

for r in tqdm(sample.itertuples(), total=len(sample), desc="Phase 2 (Cross-Lingual)"):
    gold_links = schema_links(r.db_id, r.query)
    
    for lang in CROSS_LANGS:
        question = r._asdict()[LANG_COLS[lang]]
        for cfg in top_2_configs:
            for model in MODELS:
                label = cfg_label(cfg)
                sl, k, fs, vals, vg = cfg
                if sl is None:
                    schema_text = full_schema_text(r.db_id)
                    pred_tables = {t["name"] for t in SCHEMA[r.db_id]["tables"].values()}
                else:
                    top = retrieve(r.db_id, question, sl, k)
                    schema_text, pred_tables = "\n\n".join(top["text"]), set(top["table"])
                if vals:
                    schema_text += "\n\n" + sample_values_text(r.db_id, pred_tables)
                if vg:
                    vg_note = value_grounding(r.db_id, question, pred_tables)
                    if vg_note:
                        schema_text += "\n\n" + vg_note

                try:
                    pred = generate_sql_with_retry(question, schema_text, lang=lang, few_shot=FEW_SHOT if fs else None, model=model)
                    ev = exec_match_result(r.db_id, r.query, pred)
                    pred_links = schema_links(r.db_id, pred)
                    gt, gc, pc = gold_links["tables"], gold_links["columns"], pred_links["columns"]
                    prov_cols = provided_columns(r.db_id, pred_tables)
                    
                    phase2_results.append({
                        "qid": r.qid, "db_id": r.db_id, "lang": lang, "config": label, "model": model,
                        "pred_sql": pred, "gold_sql": r.query, "question": question,
                        "complexity": sql_complexity(r.query),
                        "valid": ev["valid"], "gold_ok": ev["gold_ok"], "error": ev["error"],
                        "exec_match": ev["exec_match"],
                        "em_match": exact_match(r.query, pred),
                        "table_recall": len(gt) and len(gt & pred_tables) / len(gt) or 0,
                        "table_exact": float(bool(gt) and gt == pred_tables),
                        "col_recall": len(gc) and len(gc & prov_cols) / len(gc) or 0,
                        "col_recall_pred": len(gc) and len(gc & pc) / len(gc) or 0,
                    })
                except Exception as e:
                    pass

# Combine results
final_results = phase1_results + phase2_results
RES_CSV = ARTIFACTS / "spider_eval_results_strategy_b.csv"
pd.DataFrame(final_results).to_csv(RES_CSV, index=False)
print("\nEvaluation complete. Results saved to", RES_CSV)

results = final_results


## 12. Results — official Spider metrics (EX + EM)

In [ ]:
res = pd.DataFrame(results)
ok = res[res.gold_ok].copy()          # Spider convention: score only evaluable questions
print(f"evaluable questions: {ok['qid'].nunique()}  (gold_ok={res['gold_ok'].mean():.2%})")

summ = (ok.groupby(["config", "lang"])
        .agg(exec_acc=("exec_match", "mean"),          # official Spider EX
             em_acc=("em_match", "mean"),              # Spider-style EM
             valid_rate=("valid", "mean"),
             table_recall=("table_recall", "mean"),
             table_exact=("table_exact", "mean"),
             col_recall=("col_recall", "mean"),
             col_recall_pred=("col_recall_pred", "mean"),
             n=("exec_match", "size"))
        .round(4))

print("\n=== Spider EX / EM, by config x language ===")
print(summ.to_string())

print("\n=== EX by SQL complexity, per config ===")
print(ok.groupby(["config", "complexity"])["exec_match"].mean().round(4).to_string())

print("\n=== EM by SQL complexity, per config ===")
print(ok.groupby(["config", "complexity"])["em_match"].mean().round(4).to_string())

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

print("\n=== Multilingual Performance Gap Analysis & Statistical Significance (EX) ===")
agg = ok.groupby(["config", "lang"])["exec_match"].mean().unstack("lang")

gap = pd.DataFrame({
    "en_acc": agg.get("en"),
    "bn_acc": agg.get("bn"),
    "banglish_acc": agg.get("banglish"),
    "code_mixed_acc": agg.get("code_mixed"),
    "gap (en - bn)": agg.get("en") - agg.get("bn"),
    "gap (en - banglish)": agg.get("en") - agg.get("banglish"),
    "gap (en - code_mixed)": agg.get("en") - agg.get("code_mixed")
})
print(gap.round(4).to_string())

print("\nSmaller gaps with retrieval indicate that context helps recover multilingual translation losses.")

# --- McNemar's Test for Statistical Significance ---
print("\n=== Statistical Significance (McNemar's Test) ===")
print("Null Hypothesis: No significant difference in performance between English and the target language.")
for config in ok["config"].unique():
    print(f"\nConfig: {config}")
    df_cfg = ok[ok.config == config]
    # Create a pivot table to align questions
    pivot = df_cfg.pivot(index="qid", columns="lang", values="exec_match")
    
    for lang in ["bn", "banglish", "code_mixed"]:
        if "en" in pivot.columns and lang in pivot.columns:
            # Contingency table entries
            valid_pairs = pivot[["en", lang]].dropna()
            if len(valid_pairs) == 0:
                continue
            
            b = ((valid_pairs["en"] == True) & (valid_pairs[lang] == False)).sum()
            c = ((valid_pairs["en"] == False) & (valid_pairs[lang] == True)).sum()
            
            table = [[0, b], [c, 0]] # We only need b and c for McNemar's test
            
            try:
                result = mcnemar(table, exact=True)
                sig = "Significant" if result.pvalue < 0.05 else "Not Significant"
                print(f"  EN vs {lang.upper()}: p-value = {result.pvalue:.4f} ({sig})")
            except Exception as e:
                print(f"  EN vs {lang.upper()}: Could not compute ({e})")

print("\n=== consistency across the 4 language variants, per config ===")
cons = []
for (qid, config), g in ok.groupby(["qid", "config"]):
    outs = g["exec_match"].astype(bool).tolist()
    agree = sum(1 for a in range(len(outs)) for b in range(a + 1, len(outs))
                if outs[a] == outs[b]) / max(1, len(outs) * (len(outs) - 1) // 2)
    cons.append({"qid": qid, "config": config,
                 "n_correct": int(sum(outs)),
                 "full_consistency": float(all(outs)),
                 "pairwise_agreement": agree})
cons = pd.DataFrame(cons)
print(cons.groupby("config")
      .agg(full_consistency=("full_consistency", "mean"),
           mean_correct=("n_correct", "mean"),
           pairwise_agreement=("pairwise_agreement", "mean"))
      .round(4).to_string())


## 13. Error Analysis

In [ ]:
print("=== Error Categorization ===")
failed = res[(~res["exec_match"]) & (res["gold_ok"])]

def categorize_error(err_str):
    if not isinstance(err_str, str) or err_str.strip() == "":
        return "Logic/Semantic Error (Valid SQL but wrong result)"
    err_str = err_str.lower()
    if "syntax" in err_str or "near" in err_str:
        return "Syntax Error"
    if "no such table" in err_str:
        return "Hallucinated/Wrong Table"
    if "no such column" in err_str:
        return "Hallucinated/Wrong Column"
    return "Execution Error (Other)"

failed_df = failed.copy()
failed_df["error_category"] = failed_df["error"].apply(categorize_error)

err_summary = failed_df.groupby(["config", "lang", "error_category"]).size().unstack(fill_value=0)
print(err_summary.to_string())


## 13. Coverage report

In [ ]:
# --- Coverage report: read the saved results instead of re-running the experiment ---
cov = pd.read_csv(RES_CSV)
cov["qid"] = cov["qid"].astype(int)
sample_qids = set(cov["qid"].unique())
eval_df = vdf if EVAL_SET == "validation" else df
per_db = (eval_df.assign(in_sample=eval_df["qid"].isin(sample_qids))
                  .groupby("db_id")
                  .agg(total=("qid", "size"), covered=("in_sample", "sum"))
                  .reset_index())
per_db["coverage"] = (per_db["covered"] / per_db["total"]).round(4)
print(f"dev questions evaluated: {len(sample_qids)} / {len(eval_df)}  "
      f"dbs with >=1 sample question: {(per_db.covered > 0).sum()} / {len(per_db)}")
print(per_db.sort_values("coverage", ascending=False).to_string(index=False))